<a href="https://colab.research.google.com/github/lspnzz/granted-search-engine/blob/main/evals/notebooks/synthetic_pitch_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **A synthetic Pitch dataset for Granted.**

Notebook setup.

In [ ]:
!pip install --upgrade openai
!pip install backoff

from google.colab import userdata
from openai import OpenAI
import pandas as pd
import time
import itertools
import random

random.seed(42)  # (LS): Keeps randomness "consistent" for reproducibility

openai_key = userdata.get('openaiApiKey')
client = OpenAI(api_key=openai_key)

Diversity axes definition.

In [ ]:
industries = [
    "AI tools",
    "Healthtech",
    "Fintech",
    "Climate tech",
    "Edtech",
    "Logistics & supply chain",
    "Biotech",
    "Foodtech",
    "Creator economy",
    "E-commerce infrastructure",
    "Web3 / crypto",
    "Cybersecurity",
    "HR & recruiting",
    "Developer tools",
    "Proptech (real estate)",
    "Mobility / transportation",
    "Robotics",
    "Productivity / collaboration",
    "Media / Arts"
]

tones = [
    "YC-style concise and punchy",
    "Visionary and moonshot",
    "Corporate and formal",
    "Simple and clear"
    "Highly technical",
    "Customer-centric and empathetic",
    "Data-driven and analytical",
    "Neutral and matter-of-fact",
    "Minimalist and sparse",
]

length_categories = [
    "very short (20-40)",
    "short (50-90 words)",
    "medium (90-120 words)",
    "long (150-180 words)"
]

Create a grid of combinations and sample 100 diverse ones.

In [ ]:
all_combinations = list(itertools.product(
    industries,
    tones,
    length_categories
))

len(all_combinations)

608

We have more than 100 possible combinations, we can sample 100 unique ones.

In [ ]:
num_pitches = 100

sampled_combos = random.sample(all_combinations, num_pitches)

df_pitches = pd.DataFrame(sampled_combos, columns=[
    "industry",
    "tone",
    "length"
])

# Add an ID column
df_pitches.insert(0, "id", [f"pitch_{i:03d}" for i in range(1, num_pitches + 1)])

df_pitches.head()

,id,industry,tone,length
0,pitch_001,Climate tech,Customer-centric and empathetic,medium (90-120 words)
1,pitch_002,AI tools,Neutral and matter-of-fact,short (50-90 words)
2,pitch_003,Creator economy,Neutral and matter-of-fact,short (50-90 words)
3,pitch_004,Foodtech,Neutral and matter-of-fact,medium (90-120 words)
4,pitch_005,Foodtech,Visionary and moonshot,very short (20-40)


Create prompts for the llm.

In [ ]:
PROMPT_TEMPLATE = """Write a unique, high-quality elevator pitch for a startup.

Industry: {industry}
Tone: {tone}
Desired length: {length_category}

Requirements:
- Clearly state the problem, the target user, and the solution.
- Highlight what makes this startup differentiated from existing solutions.
- Match the requested tone.
- Do NOT mention that you were given these fields; just write the pitch as if for a real startup.
"""

def make_prompt(row):
    return PROMPT_TEMPLATE.format(
        industry=row["industry"],
        tone=row["tone"],
        length_category=row["length"]
    )

df_pitches["prompt"] = df_pitches.apply(make_prompt, axis=1)

# Inspect a few prompts
df_pitches[["id", "prompt"]].head().iloc[0]["prompt"]

'Write a unique, high-quality elevator pitch for a startup.\n\nIndustry: Climate tech\nTone: Customer-centric and empathetic\nDesired length: medium (90-120 words)\n\nRequirements:\n- Clearly state the problem, the target user, and the solution.\n- Highlight what makes this startup differentiated from existing solutions.\n- Match the requested tone.\n- Do NOT mention that you were given these fields; just write the pitch as if for a real startup.\n'

Define the api call.

In [ ]:
import backoff

MODEL_NAME = "gpt-4.1-mini"

@backoff.on_exception(backoff.expo, Exception, max_tries=5)
def generate_pitch(prompt: str) -> str:
    """
    Calls the OpenAI Responses API with the prompt and returns plain text.
    """
    response = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
        max_output_tokens=350,
        temperature=0.7
    )
    return response.output_text.strip()

Generate pitches.

In [ ]:
df_pitches["pitch_text"] = None

for idx, row in df_pitches.iterrows():
    print(f"Generating pitch {row['id']}...")

    try:
        pitch = generate_pitch(row["prompt"])
    except Exception as e:
        print("Error during generation:", e)
        pitch = None

    df_pitches.at[idx, "pitch_text"] = pitch

    time.sleep(0.2)  # small delay to be polite to the API

print("Done generating all pitches!")

Generating pitch pitch_001...
Generating pitch pitch_002...
Generating pitch pitch_003...
Generating pitch pitch_004...
Generating pitch pitch_005...
Generating pitch pitch_006...
Generating pitch pitch_007...
Generating pitch pitch_008...
Generating pitch pitch_009...
Generating pitch pitch_010...
Generating pitch pitch_011...
Generating pitch pitch_012...
Generating pitch pitch_013...
Generating pitch pitch_014...
Generating pitch pitch_015...
Generating pitch pitch_016...
Generating pitch pitch_017...
Generating pitch pitch_018...
Generating pitch pitch_019...
Generating pitch pitch_020...
Generating pitch pitch_021...
Generating pitch pitch_022...
Generating pitch pitch_023...
Generating pitch pitch_024...
Generating pitch pitch_025...
Generating pitch pitch_026...
Generating pitch pitch_027...
Generating pitch pitch_028...
Generating pitch pitch_029...
Generating pitch pitch_030...
Generating pitch pitch_031...
Generating pitch pitch_032...
Generating pitch pitch_033...
Generating

In [ ]:
df_pitches[["id", "industry", "pitch_text"]].head()

,id,industry,pitch_text
0,pitch_001,Climate tech,"Every day, millions of households struggle to ..."
1,pitch_002,AI tools,Many professionals struggle with managing over...
2,pitch_003,Creator economy,Creators today struggle to monetize diverse co...
3,pitch_004,Foodtech,FoodChoice addresses the growing challenge fac...
4,pitch_005,Foodtech,We’re revolutionizing global nutrition by crea...


Save the dataset.

In [ ]:
df_pitches.to_csv("startup_pitches.csv", index=False)